# 🔍 Plataforma RCA — Detección de Anomalías en Microservicios (Fase 1)
**Descripción:** Este notebook contiene el pipeline definitivo para la detección no supervisada de anomalías e identificación de causa raíz (RCA) utilizando logs unificados por `traceId` procedentes del stack PLG (Loki).
**Modelo:** Isolation Forest + TF-IDF (NLP) + Standard Scaling.

---

In [1]:
import seaborn as sns
import pandas as pd
import numpy as np
import shap
import re
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import make_scorer, f1_score
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix

%matplotlib inline

/opt/homebrew/Caskroom/miniforge/base/envs/rca/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Carga de Datos y Consolidación de Logs
En esta sección importamos los logs en crudo extraídos de Loki. Para reconstruir el ciclo de vida de cada petición y evitar la pérdida de contexto por condiciones de carrera en los hilos de logging:
1. Eliminamos las filas donde `event_type, http_uri y duration_ms` son nulos de manera conjunta. No serán de ayuda esas filas.
2. Eliminamos los endpoints de `CHAOS` generados para conseguir el dataset.
3. Creamos una nueva nueva feature `log_count` donde contamos los logs que se han generado por un mismo trace-id. Esta información le servirá al modelo para poder discernir entre una petición anómala y una normal
4. Juntamos los logs por su `traceid` para tener un único log con toda la información. Debemos de arreglar la columna `level` ya que fruto de juntar los logs, puede contener varios datos juntos

In [9]:
df_1 = pd.read_csv("../datasets/01_logs_dataset.csv")
df_2 = pd.read_csv("../datasets/02_logs_dataset.csv")
df = pd.concat([df_1, df_2], ignore_index=True)
df.head()

# Elimina filas donde event_type, http_uri y duration_ms son todos nulos Y quitamos outcome porque aporta redundancia
df = df.dropna(subset=["event_type", "http_uri", "duration_ms"], how="all")
df = df.drop(columns=["outcome"])

# Elimina también los endpoints de caos
df = df[~df["http_uri"].str.contains("/chaos/", na=False)]

# eliminamos las columnas que ya no sirven
cdf = df.drop(columns=["http_uri"])
print(cdf["uri_servicio"].value_counts())
print(len(cdf))

KeyError: 'uri_servicio'

In [3]:
# Contamos los logs que genero una misma peticion (conocemos que si son 2 suele ser fallo, puede ser una buena medida)
cdf['log_count'] = cdf.groupby(['traceId', 'service'])['traceId'].transform('count')
cdf['log_count'].value_counts()

log_count
2    2952
1    2313
Name: count, dtype: int64

In [6]:
# 1. Asegurarnos de que el dataframe esté ordenado por tiempo
cdf = cdf.sort_values(by=['@timestamp'])

# 2. PROPAGAR EL TRACEID: 
# Agrupamos por tiempo exacto y servicio, y propagamos el traceId hacia atrás (bfill) y hacia adelante (ffill)
# Así, la fila que tiene NaN adoptará el traceId '6a4bac01...' de su fila hermana.
cdf['traceId'] = cdf.groupby(['@timestamp', 'service'])['traceId'].transform(lambda x: x.bfill().ffill())

# Opcional pero recomendado: si queda algún log de sistema puro que no pertenece a ninguna 
# petición (sigue teniendo traceId = NaN), lo filtramos para no meter ruido al modelo.
cdf = cdf.dropna(subset=['traceId'])

# 3. EL DICCIONARIO DE AGREGACIÓN (El que ya teníamos)
agregaciones = {
    '@timestamp': 'min',  
    'level': lambda x: ', '.join(x.dropna().astype(str).unique()),
    'event_type': lambda x: ', '.join(x.dropna().astype(str).unique()),
    'http_method': 'first',
    'http_status': lambda x: ', '.join(x.dropna().astype(str).unique()),
    'duration_ms': 'sum', 
    'error_type': lambda x: ' | '.join(x.dropna().astype(str).unique()),
    'error_message': lambda x: ' | '.join(x.dropna().astype(str).unique()),
    'error_origin': lambda x: ' | '.join(x.dropna().astype(str).unique()),
    'uri_servicio': 'first',
    'log_count':'max'
}

# 4. AGRUPACIÓN FINAL
df_clear = cdf.groupby(['traceId', 'service']).agg(agregaciones).reset_index()

# arreglamos que ahora haya varios level en uno mismo
def max_level(comb_level):
    level = str(comb_level).upper()
    if 'ERROR' in level:
        return 'ERROR'
    elif 'WARN' in level:
        return 'WARN'
    else:
        return 'INFO'

df_clear['level'] = df_clear['level'].apply(max_level)

# comprobamos el df
df_clear.head()

,traceId,service,@timestamp,level,event_type,http_method,http_status,duration_ms,error_type,error_message,error_origin,uri_servicio,log_count
0,6a476cca227cf2c2a7778eea6814c767,api-autenticacion,2026-07-03T08:03:23.032Z,INFO,HTTP_REQUEST,POST,201.0,363.0,,,,autenticacion,1
1,6a476ccf073988540242f264230c10a8,api-inventario,2026-07-03T08:03:27.935Z,INFO,HTTP_REQUEST,POST,200.0,158.0,,,,inventario,1
2,6a476cd501fdf3e29f19f31f7e7dca68,api-inventario,2026-07-03T08:03:33.981Z,INFO,HTTP_REQUEST,GET,200.0,92.0,,,,inventario,1
3,6a476cd735ddf16a117971907acb4d4e,api-inventario,2026-07-03T08:03:36.098Z,INFO,HTTP_REQUEST,POST,200.0,40.0,,,,inventario,1
4,6a476cd735ddf16a117971907acb4d4e,api-pedidos,2026-07-03T08:03:36.202Z,INFO,HTTP_REQUEST,POST,200.0,297.0,,,,pedidos,1


#### 1.1 Ingeniería de Características
1. Creamos la columna `tiene_error_5xx` que tomarán valores binarios para indicar cuando una petición tiene un error de la familia 500. De esta forma, el modelo entiende de manera más fácil de qué se trata
2. Creamos una feature nueva llamada `duracion_relativa` que indica cuánto se desvía la latencia partiendo de un baseline calculado por la media de aquellas peticiones que van a un determinado servicio
3. Creamos una feature nueva llamada `cascada` que indicara con `0´s` cuando el log NO ha sido generado por un error en casacada y `1´s` cuando si lo haya sido
4. Eliminamos las columnas `@timestamp, , service y uri_servicio"`